# Heel-video shoe tracker — SAM 2 validation

This notebook answers one question cheaply: **when you twist your foot to show the shoe, does SAM 2 keep a clean outline on the shoe the whole time?**

If it does, replacing the shoe by compositing is realistic. If the outline slips or falls apart, the twist beat needs the heavier 3D approach.

**Two rules that make or break this test**
1. **Initialize on the TWIST frame, not the walk-in.** Pick a frame where your foot is planted, still, and turned to show the heel. The blurry walk-in/out frames are the hardest for SAM 2 — start on a clean one and let it track outward from there.
2. **Box the shoe, don't click the leg.** A tight rectangle around just the shoe is far more reliable than a single point (a point near the ankle grabs the whole leg).

**How to run it**
1. `Runtime` -> `Change runtime type` -> **GPU** -> Save.
2. Optional but recommended: `File` -> `Save a copy in Drive`, so your edits stick (the "saving failed" banner just means this copy is read-only).
3. Run top to bottom. Upload one short clip, pick the twist frame, box the shoe, watch the preview.

## 1. Confirm you have a GPU
If this shows nothing, set the runtime to GPU (see above).

In [ ]:
!nvidia-smi

## 2. Install SAM 2 and download the model
A couple of minutes the first time.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
print("Done. Model downloaded.")

## 3. Upload one of your clips
Run, then pick a video. Keep the first test short.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = "/content/sam2/" + list(uploaded.keys())[0]
print("Uploaded:", VIDEO_PATH)

## 4. Break the video into frames

In [ ]:
import os, glob
FRAME_DIR = "/content/frames"
os.system(f"rm -rf {FRAME_DIR}")
os.makedirs(FRAME_DIR, exist_ok=True)
os.system(f'ffmpeg -loglevel error -i "{VIDEO_PATH}" -q:v 2 -start_number 0 "{FRAME_DIR}/%05d.jpg"')
frames = sorted(glob.glob(f"{FRAME_DIR}/*.jpg"))
print(f"Extracted {len(frames)} frames.")

## 5. Pick the TWIST frame

Set `INIT_FRAME` to a frame index where your foot is **planted and turned to show the shoe** — still and sharp, not the blurry walk-in. Re-run to scrub to different frames until you find a good one. Note the x,y grid — you'll box the shoe next.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

INIT_FRAME = 0          # <-- change this to your twist frame (0 .. total-1)

print("total frames:", len(frames), "-> valid INIT_FRAME is 0 to", len(frames) - 1)
img = Image.open(frames[INIT_FRAME]); w, h = img.size
fig, ax = plt.subplots(figsize=(11, 11 * h / w))
ax.imshow(img)
ax.set_xticks(range(0, w, max(1, w // 20)))
ax.set_yticks(range(0, h, max(1, h // 20)))
ax.grid(color="cyan", linestyle=":", linewidth=0.6, alpha=0.7)
ax.set_title(f"frame {INIT_FRAME} of {len(frames)}  —  {w} wide x {h} tall")
plt.show()

## 6. Box the shoe and preview the mask

Read a rectangle around **just the shoe** off the grid above: `BOX = [x_left, y_top, x_right, y_bottom]`. Keep it snug — a little padding is fine, but don't include the whole foot or floor.

This previews the mask on your chosen frame *before* tracking the whole clip. Re-run and adjust the box until the red covers the shoe and nothing else. (If a box alone isn't enough, add positive/negative points too.)

In [ ]:
import numpy as np
import torch
from sam2.build_sam import build_sam2_video_predictor

# ---- box tightly around JUST the shoe ----
BOX = [400, 900, 700, 1200]          # [x_left, y_top, x_right, y_bottom]
# ---- optional extra hints (usually leave empty) ----
POSITIVE_POINTS = []                 # points ON the shoe, e.g. [[550, 1050]]
NEGATIVE_POINTS = []                 # points to EXCLUDE (leg/floor), e.g. [[600, 700]]
# ------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    "checkpoints/sam2.1_hiera_large.pt",
    device=device,
)

state = predictor.init_state(video_path=FRAME_DIR)
predictor.reset_state(state)

kw = dict(inference_state=state, frame_idx=INIT_FRAME, obj_id=1,
          box=np.array(BOX, dtype=np.float32))
pts = list(POSITIVE_POINTS) + list(NEGATIVE_POINTS)
if pts:
    kw["points"] = np.array(pts, dtype=np.float32)
    kw["labels"] = np.array([1] * len(POSITIVE_POINTS) + [0] * len(NEGATIVE_POINTS),
                            dtype=np.int32)

_, obj_ids, mask_logits = predictor.add_new_points_or_box(**kw)
mask = (mask_logits[0] > 0).cpu().numpy().squeeze()

img = Image.open(frames[INIT_FRAME]); w, h = img.size
fig, ax = plt.subplots(figsize=(11, 11 * h / w))
ax.imshow(img)
overlay = np.zeros((*mask.shape, 4)); overlay[mask] = [1, 0, 0, 0.5]
ax.imshow(overlay)
ax.add_patch(plt.Rectangle((BOX[0], BOX[1]), BOX[2] - BOX[0], BOX[3] - BOX[1],
                           fill=False, edgecolor="yellow", linewidth=2))
for px, py in POSITIVE_POINTS: ax.plot(px, py, "go", markersize=10)
for nx, ny in NEGATIVE_POINTS: ax.plot(nx, ny, "x", color="white", markersize=10)
ax.set_title("Red = shoe mask. Adjust BOX until it covers the shoe and nothing else.")
plt.show()

## 7. Track the shoe through the whole clip

Once the box preview looks right, run this. It tracks **outward in both directions** from your twist frame, so earlier (walk-in) and later (walk-out) frames are all covered.

In [ ]:
masks = {}
for fi, ids, ml in predictor.propagate_in_video(state, start_frame_idx=INIT_FRAME, reverse=False):
    masks[fi] = (ml[0] > 0).cpu().numpy().squeeze()
for fi, ids, ml in predictor.propagate_in_video(state, start_frame_idx=INIT_FRAME, reverse=True):
    masks[fi] = (ml[0] > 0).cpu().numpy().squeeze()
print(f"Tracked the shoe across {len(masks)} frames.")

## 8. Build the preview video

Tints the tracked shoe red in every frame and stitches an MP4. **Watch the twist**: the red should stay glued to the shoe as your foot rotates.

In [ ]:
import cv2

out_dir = "/content/overlay_frames"
os.system(f"rm -rf {out_dir}"); os.makedirs(out_dir, exist_ok=True)
for i, fpath in enumerate(frames):
    frame = cv2.imread(fpath)
    m = masks.get(i)
    if m is not None and m.any():
        red = np.zeros_like(frame); red[:, :, 2] = 255
        frame = np.where(m[..., None], (0.5 * frame + 0.5 * red).astype(np.uint8), frame)
    cv2.imwrite(f"{out_dir}/{i:05d}.jpg", frame)
os.system(f'ffmpeg -loglevel error -y -framerate 24 -i "{out_dir}/%05d.jpg" '
          f'-c:v libx264 -pix_fmt yuv420p /content/preview.mp4')
print("Wrote /content/preview.mp4")

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("/content/preview.mp4", "rb").read()).decode()
HTML(f'<video width=500 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## How to read the result

- **Red stays locked on the shoe through the twist** -> compositing is viable; that red region is exactly where a new shoe would be painted in.
- **Red slips onto the leg/floor, shrinks, or flickers during the twist** -> that beat likely needs the 3D-rendering approach; the walk-in/out can still use this.

If tracking drifts, go back to cell 5 (try a different twist frame), then cell 6 (tighten the box, add a negative point on the leg), and re-run 6 -> 7 -> 8. Tell me what the twist looks like and we'll pick the build path.